# Inspect Generated Target Corpora

This notebook helps debug and characterize procedurally generated target datasets.

It can:
- load a target manifest
- preview individual power and importance maps
- summarize category and beam-count distributions
- characterize support coverage and power-map ranges across the corpus


In [ ]:
from __future__ import annotations

import random
import sys
from collections import Counter
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "generation").exists() and (ROOT.parent / "generation").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import torch
import yaml

from generation.target_corpus import generateTargetCorpus

MANIFEST_PATH = ROOT / "data/targets/simple/manifest.yaml"
RNG = random.Random(7)

plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["image.origin"] = "lower"
plt.rcParams["axes.grid"] = False

print(f"Project root: {ROOT}")
print(f"Manifest path: {MANIFEST_PATH}")

## Optional: Regenerate The Corpus With Progress

Run this cell when you want to rebuild the corpus directly from the notebook and watch a notebook-friendly progress bar.

In [ ]:
GENERATE_FROM_NOTEBOOK = False

if GENERATE_FROM_NOTEBOOK:
    generateTargetCorpus(
        ROOT / "configs/target_corpus.yaml",
        workers="auto",
        showProgress=True,
        progressDescription="Notebook corpus build",
    )

In [ ]:
with MANIFEST_PATH.open("r", encoding="utf-8") as handle:
    manifest = yaml.safe_load(handle)

records = manifest["records"]
print("Corpus metadata")
print(f"  count: {manifest['count']}")
print(f"  powerMode: {manifest.get('powerMode')}")
print(f"  maxHotspots: {manifest.get('maxHotspots')}")
print(f"  grid: {manifest.get('grid')}")
print(f"  curriculum counts: {manifest.get('curriculum', {}).get('counts')}")

In [ ]:
def resolve_record_path(record: dict) -> Path:
    path = Path(record["path"])
    return path if path.is_absolute() else MANIFEST_PATH.parent / path


def load_target(record_or_index: int | dict):
    record = records[record_or_index] if isinstance(record_or_index, int) else record_or_index
    return torch.load(resolve_record_path(record), weights_only=False)


def power_display_map(target) -> tuple[torch.Tensor, str]:
    power = target.powerMap.detach().cpu()
    if manifest.get("powerMode") == "db" or torch.any(power < 0):
        return power, "Power (dB)"
    return power, "Power (normalized linear)"


def preview_target(index: int) -> None:
    record = records[index]
    target = load_target(record)
    power_map, power_label = power_display_map(target)
    importance = target.importanceMap.detach().cpu()
    lat = target.searchLatitudes.detach().cpu()
    lon = target.searchLongitudes.detach().cpu()
    extent = [lon.min().item(), lon.max().item(), lat.min().item(), lat.max().item()]

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    im0 = axes[0].imshow(power_map, extent=extent, aspect="auto", cmap="viridis")
    axes[0].set_title(f"Power Map | idx={index} | {record['category']}")
    axes[0].set_xlabel("Longitude")
    axes[0].set_ylabel("Latitude")
    fig.colorbar(im0, ax=axes[0], pad=0.02, label=power_label)

    im1 = axes[1].imshow(importance, extent=extent, aspect="auto", cmap="magma")
    hotspots = target.hotspotCoordinates.detach().cpu()
    axes[1].scatter(hotspots[:, 1], hotspots[:, 0], c="cyan", s=25, label="hotspots")
    axes[1].set_title("Importance Map")
    axes[1].set_xlabel("Longitude")
    axes[1].set_ylabel("Latitude")
    axes[1].legend(loc="upper right")
    fig.colorbar(im1, ax=axes[1], pad=0.02, label="Importance")
    plt.show()

    print({
        "index": index,
        "shape": power_map.shape,
        "category": record.get("category"),
        "beamCount": record.get("beamCount"),
        "zoneCount": record.get("zoneCount"),
        "power_min": float(power_map.min()),
        "power_max": float(power_map.max()),
        "importance_nonzero_fraction": float((importance > 0).float().mean()),
        "hotspot_count": int(hotspots.shape[0]),
    })


def preview_random_target() -> None:
    preview_target(RNG.randrange(len(records)))

In [ ]:
category_counts = Counter(record.get("category", "unknown") for record in records)
beam_counts = Counter(record.get("beamCount", 0) for record in records)

print("Category counts:", dict(category_counts))
print("Beam counts:", dict(sorted(beam_counts.items())))

In [ ]:
stats = []
for index, record in enumerate(records):
    target = load_target(record)
    power_map = target.powerMap.detach().cpu()
    importance = target.importanceMap.detach().cpu()
    stats.append(
        {
            "index": index,
            "category": record.get("category", "unknown"),
            "beamCount": int(record.get("beamCount", 0) or 0),
            "support_fraction": float((importance > 0).float().mean()),
            "importance_mean": float(importance.mean()),
            "power_min": float(power_map.min()),
            "power_max": float(power_map.max()),
            "hotspot_count": int(target.hotspotCoordinates.shape[0]),
        }
    )

print(f"Collected stats for {len(stats)} targets")
stats[:3]

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

axes[0].bar(category_counts.keys(), category_counts.values(), color="#2b8cbe")
axes[0].set_title("Category Distribution")
axes[0].tick_params(axis="x", rotation=20)

axes[1].bar([str(k) for k in sorted(beam_counts)], [beam_counts[k] for k in sorted(beam_counts)], color="#7bccc4")
axes[1].set_title("Beam Count Distribution")
axes[1].set_xlabel("Beam count")

axes[2].hist([row["support_fraction"] for row in stats], bins=12, color="#fdae6b")
axes[2].set_title("Support Fraction")
axes[2].set_xlabel("Non-zero importance fraction")

axes[3].hist([row["power_max"] for row in stats], bins=12, color="#74c476")
axes[3].set_title("Peak Power Values")
axes[3].set_xlabel("Max power")

plt.tight_layout()
plt.show()

In [ ]:
preview_target(0)

In [ ]:
preview_random_target()